<a href="https://colab.research.google.com/github/thehmfpk/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

```markdown
**Plain Words Contract:**

*   **One row = one what?** Each row in our primary dataset (`fact_content_daily_performance`) represents **one content item's performance on a specific client, on a specific report date.** This is the `report_date × client × content` grain.
*   **Which table(s)?** We will primarily use `fact_content_daily_performance` (or its `_sample` version for initial exploration) from the `hf://datasets/FlyRank/internship-warehouse` dataset.
*   **Which time window?** The data covers **2025-01-27 to 2026-06-30**. For verification and feature engineering, we will focus on a **mid-panel month, specifically March 2026** (2026-03-01 to 2026-03-31).
*   **What to predict (label)?** The label we aim to predict is whether a content item is **`is_declining_label`**.
*   **One thing excluded?** We will explicitly exclude **`trend_direction` and `trend_pct`** as features due to direct leakage with the label `is_declining_label`.
```

In [4]:
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
from google.colab import userdata
import os
from datetime import datetime

# --- Contract declarations (unchanged) ---
unit_of_analysis = "one content item's performance on a specific client, on a specific report date"
primary_key_columns = ['report_date', 'client_hash_id', 'content_hash_id']
data_start_date_str = "2026-03-01"
data_end_date_str = "2026-03-31"
data_start_date = datetime.strptime(data_start_date_str, '%Y-%m-%d').date()
data_end_date = datetime.strptime(data_end_date_str, '%Y-%m-%d').date()

print(f"Declared Unit of Analysis: {unit_of_analysis}")
print(f"Declared Primary Key Column(s): {primary_key_columns}")
print(f"Declared Data Time Window: From {data_start_date_str} to {data_end_date_str}")

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

try:
    # --- STEP 1: find the actual March-2026 partition files, don't guess the path ---
    print("\nDiscovering repo files to find the March 2026 partition (avoids downloading all 17 months)...")
    api = HfApi(token=HF_TOKEN)
    all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
    march_files = [f for f in all_files
                   if "fact_content_daily_performance" in f
                   and "sample" not in f  # exclude the _sample table, wrong table entirely
                   and "2026-03" in f]

    if not march_files:
        print("[WARNING] No files matched 'fact_content_daily_performance' + '2026-03'.")
        print("Printing all fact_content_daily_performance files so you can see the real naming pattern:")
        for f in [x for x in all_files if "fact_content_daily_performance" in x and "sample" not in x][:20]:
            print(" ", f)
        raise ValueError("Fix the march_files filter above to match the real path pattern, then re-run.")

    print(f"Found {len(march_files)} file(s) for March 2026:")
    for f in march_files:
        print(" ", f)

    # --- STEP 2: load ONLY those files - no 78.8M-row download, no filter pass ---
    print("\nLoading only the March 2026 partition...")
    dataset = load_dataset(
        "FlyRank/internship-warehouse",
        data_files={"train": march_files},
        split="train"
    )
    print(f"Loaded {len(dataset)} rows, {len(dataset.column_names)} columns directly (no full-table filter needed).")

    # --- STEP 3: select only needed columns before converting to pandas (cuts memory further) ---
    if 'fields' not in globals():
        print("[WARNING] 'fields' dict not found - run the field-categorization cell before this one.")
        fields = {'label': [], 'context': ['report_date', 'client_hash_id', 'content_hash_id'],
                   'feature': ['gsc_clicks', 'gsc_impressions']}

    all_required_cols = []
    for category in ['label', 'feature', 'context']:
        all_required_cols.extend(fields.get(category, []))
    for col in primary_key_columns:
        if col not in all_required_cols:
            all_required_cols.append(col)
    if 'report_date' not in all_required_cols:
        all_required_cols.append('report_date')

    available_dataset_cols = set(dataset.column_names)
    cols_to_select = [col for col in all_required_cols if col in available_dataset_cols]
    print(f"Selecting {len(cols_to_select)} of {len(dataset.column_names)} columns: {', '.join(cols_to_select)}")
    dataset = dataset.select_columns(cols_to_select)

    # --- STEP 4: convert to pandas, then immediately downcast dtypes to shrink memory footprint ---
    print("Converting to Pandas DataFrame...")
    df = dataset.to_pandas()

    if 'report_date' in df.columns:
        df['report_date'] = pd.to_datetime(df['report_date'])
    for col in ['client_hash_id', 'content_hash_id']:
        if col in df.columns:
            df[col] = df[col].astype('category')  # big memory win on repeated hash strings
    for col in df.select_dtypes(include='float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include='int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    print(f"Successfully loaded {len(df)} rows. Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    print(f"\nFirst 5 rows:\n{df.head()}")

except Exception as e:
    print(f"\n[ERROR] Failed to load data: {e}")
    print("If this still crashes with RAM errors: Runtime -> Change runtime type -> pick a High-RAM option,")
    print("or trim 'fields[\"feature\"]' down to only the columns you actually need for this pass.")
    df = pd.DataFrame()

Declared Unit of Analysis: one content item's performance on a specific client, on a specific report date
Declared Primary Key Column(s): ['report_date', 'client_hash_id', 'content_hash_id']
Declared Data Time Window: From 2026-03-01 to 2026-03-31

Discovering repo files to find the March 2026 partition (avoids downloading all 17 months)...
Found 1 file(s) for March 2026:
  fact_content_daily_performance/month=2026-03/data_0.parquet

Loading only the March 2026 partition...


Generating train split: 0 examples [00:00, ? examples/s]

Loaded 9841378 rows, 30 columns directly (no full-table filter needed).
Selecting 30 of 30 columns: gsc_clicks, gsc_impressions, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available
Converting to Pandas DataFrame...
Successfully loaded 9841378 rows. Memory usage: 1393.9 MB

First 5 rows:
   gsc_clicks  gsc_impressions  gsc_sum_position  gsc_avg_position  \
0           0               20                67          3.350000   
1           0                1                 0          0.000000   
2           1              125               616          4.928000   
3           0                7                28 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
import pandas as pd

fields = {
    'label': [
    ],
    'excluded': [
    ],
    'context': [
        'report_date',
        'client_hash_id',
        'content_hash_id',
        'client_has_gsc',
        'client_has_ga4',
        'gsc_data_available',
        'ga4_data_available'
    ],
    'feature': [
        'gsc_clicks',
        'gsc_impressions',
        'gsc_sum_position',
        'gsc_avg_position',
        'ga4_pageviews',
        'ga4_sessions',
        'ga4_users',
        'ga4_engaged_sessions',
        'ga4_total_engagement_sec',
        'sessions_organic',
        'sessions_direct',
        'sessions_referral',
        'sessions_social',
        'sessions_paid',
        'sessions_ai',
        'ai_chatgpt',
        'ai_perplexity',
        'ai_gemini',
        'ai_copilot',
        'ai_claude',
        'ai_meta',
        'ai_other',
        'scroll_events'
    ]
}

# Display the categorized fields
print("--- Field Categorization ---")
for category, field_list in fields.items():
    if field_list:
        print(f"{category.capitalize()}: {', '.join(field_list)}")
    else:
        print(f"{category.capitalize()}: (none yet)")

# Basic validation (optional, but good practice)
all_fields = []
for category in fields:
    all_fields.extend(fields[category])

if len(all_fields) != len(set(all_fields)):
    print("\n[WARNING] Duplicate fields found across categories!")

# Placeholder to ensure df exists for subsequent steps, even if empty
if 'df' not in locals():
    df = pd.DataFrame()

# Check if all listed fields are present in the (potentially empty) DataFrame
if not df.empty:
    missing_fields = [f for f in all_fields if f not in df.columns]
    if missing_fields:
        print(f"\n[WARNING] The following fields are listed but not found in the DataFrame: {', '.join(missing_fields)}")
        print("If this list includes GA4/session/AI columns, the loading cell likely ran BEFORE this")
        print("cell and used its 2-column fallback. Re-run the loading cell now that `fields` is defined.")
    else:
        print("\nAll listed fields are present in the DataFrame.")
else:
    print("\nDataFrame 'df' is currently empty or not loaded. Run the loading cell after this one.")

--- Field Categorization ---
Label: (none yet)
Excluded: (none yet)
Context: report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available
Feature: gsc_clicks, gsc_impressions, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events

All listed fields are present in the DataFrame.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
import pandas as pd

print("--- Data Verification Queries ---")

if 'df' not in locals() or df.empty:
    print("\n[ERROR] DataFrame 'df' is empty or not loaded. Cannot perform verification queries.")
    print("Please ensure the data loading cell (c0983f8f) ran successfully and data was loaded.")
else:
    print(f"\nTotal rows in DataFrame: {len(df)}")
    print(f"Columns in DataFrame: {', '.join(df.columns)}")

    # 1. Unit of Analysis Verification (grain): Check for uniqueness of primary keys
    print("\n1. Unit of Analysis Verification (grain):")
    if primary_key_columns and all(col in df.columns for col in primary_key_columns):
        num_duplicates = df.duplicated(subset=primary_key_columns).sum()
        if num_duplicates == 0:
            print(f"  All combinations of {primary_key_columns} are unique. Unit of analysis is correctly at grain.")
        else:
            print(f"  Found {num_duplicates} duplicate rows based on {primary_key_columns}. Expected unique grain.")
            print("    Example duplicates (first 5):\n", df[df.duplicated(subset=primary_key_columns, keep=False)].sort_values(by=primary_key_columns).head())
    else:
        print(f"  Skipping primary key uniqueness check: primary_key_columns ({primary_key_columns}) is not defined or not all columns exist in df.")

    # 2. Counts + Date Span Verification:
    print("\n2. Counts + Date Span Verification:")
    expected_min_rows = 1000  # Example: Set a reasonable minimum based on expected dataset size
    if len(df) >= expected_min_rows:
        print(f"  Row count ({len(df)}) meets or exceeds expected minimum of {expected_min_rows}.")
    else:
        print(f"  Row count ({len(df)}) is less than expected minimum of {expected_min_rows}. Investigate data source or filters.")

    if 'report_date' in df.columns and pd.api.types.is_datetime64_any_dtype(df['report_date']):
        min_date = df['report_date'].min()
        max_date = df['report_date'].max()
        n_content = df['content_hash_id'].nunique() if 'content_hash_id' in df.columns else None
        n_clients = df['client_hash_id'].nunique() if 'client_hash_id' in df.columns else None

        print(f"  Observed data range: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
        print(f"  Declared data range: {data_start_date} to {data_end_date}")
        if n_content is not None:
            print(f"  Distinct content items: {n_content}")
        if n_clients is not None:
            print(f"  Distinct clients: {n_clients}")

        matches_declared_window = (pd.to_datetime(min_date) == pd.to_datetime(data_start_date) and
                                    pd.to_datetime(max_date) == pd.to_datetime(data_end_date))
        if matches_declared_window:
            print(f"  Data's time window perfectly matches the declared window.")
        elif (pd.to_datetime(min_date) >= pd.to_datetime(data_start_date) and
              pd.to_datetime(max_date) <= pd.to_datetime(data_end_date)):
            print(f"  Data's time window is within the declared window, but doesn't cover the full range.")
        else:
            print(f"  Data's time window falls outside the declared window ({data_start_date} - {data_end_date}).")

        expected_days = (pd.to_datetime(data_end_date) - pd.to_datetime(data_start_date)).days + 1
        actual_unique_days = df['report_date'].nunique()
        if actual_unique_days == expected_days:
            print(f"  All {expected_days} days present in the declared time window.")
        else:
            print(f"  Only {actual_unique_days} unique days found out of {expected_days} expected days. Possible missing days.")
    else:
        print("  Skipping time window check: 'report_date' column not found or not in datetime format.")

    # 3. Availability Verification (IS TRUE) - REQUIRED per the task rubric, was missing before.
    print("\n3. Availability Verification (IS TRUE):")
    for flag_col in ['gsc_data_available', 'ga4_data_available']:
        if flag_col in df.columns:
            total = len(df)
            available = df[df[flag_col] == True].shape[0]
            print(f"  {flag_col} IS TRUE: {available} / {total} rows survive ({available/total:.1%})")
        else:
            print(f"  Skipping {flag_col}: column not found in df. Add it to fields['context'] and reload if you need this check.")

    # Bonus: gsc_avg_position = 0 means "no data", not rank zero - don't average it in blind.
    if 'gsc_avg_position' in df.columns:
        zero_position_rows = (df['gsc_avg_position'] == 0).sum()
        print(f"  gsc_avg_position == 0 ('no data', not rank 0): {zero_position_rows} rows "
              f"({zero_position_rows/len(df):.1%}) - exclude these before averaging position.")

    # 4. Missing Values Verification:
    print("\n4. Missing Values Verification:")
    columns_to_check_for_missing = []
    for category in ['label', 'feature', 'context']:
        if category in fields:
            columns_to_check_for_missing.extend([f for f in fields[category] if f in df.columns])

    if columns_to_check_for_missing:
        missing_data = df[columns_to_check_for_missing].isnull().sum()
        missing_data = missing_data[missing_data > 0]

        if missing_data.empty:
            print("  No NaN values found in label, feature, and context columns.")
            print("  NOTE: this does NOT mean GA4 columns are clean for every client - per flyrank-data,")
            print("  GA4 columns are zero-FILLED (not NaN) before a client's ga4_data_start, so a clean")
            print("  isnull() result can still hide meaningless zeros. That's what the IS TRUE check above catches.")
        else:
            print("  Missing (NaN) values found in the following columns:")
            for col, count in missing_data.items():
                print(f"    - {col}: {count} missing values ({count/len(df):.2%})")
    else:
        print("  Skipping missing values check: No label, feature, or context columns identified in the DataFrame.")

print("\n--- Verification queries complete. Analyze the results above! ---")

--- Data Verification Queries ---

Total rows in DataFrame: 9841378
Columns in DataFrame: gsc_clicks, gsc_impressions, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available

1. Unit of Analysis Verification (grain):
  All combinations of ['report_date', 'client_hash_id', 'content_hash_id'] are unique. Unit of analysis is correctly at grain.

2. Counts + Date Span Verification:
  Row count (9841378) meets or exceeds expected minimum of 1000.
  Observed data range: 2026-03-01 to 2026-03-31
  Declared data range: 2026-03-01 to 2026-03-31
  Distinct content items: 331437
  Distinct clients: 55
  Data's time window p

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [9]:
import pandas as pd

print("--- Data Limitations Documentation ---")

history_depth_note = ""
if 'df' in locals() and not df.empty and 'client_hash_id' in df.columns and 'report_date' in df.columns:
    days_per_client = df.groupby('client_hash_id')['report_date'].nunique()
    min_days, max_days = days_per_client.min(), days_per_client.max()
    thin_clients = (days_per_client < days_per_client.median() / 2).sum()
    history_depth_note = (f"Observed: within March 2026, per-client day coverage ranges from "
                           f"{min_days} to {max_days} days out of the month; {thin_clients} of "
                           f"{len(days_per_client)} clients have less than half the median coverage.")
else:
    history_depth_note = "Could not compute -- df/client_hash_id/report_date not available when this cell ran."

# 2. GSC availability: pull the real fraction from what section 3 already measured.
availability_note = ""
if 'df' in locals() and not df.empty and 'gsc_data_available' in df.columns:
    gsc_avail_frac = (df['gsc_data_available'] == True).mean()
    availability_note = f"Observed: {gsc_avail_frac:.1%} of rows in this slice have gsc_data_available == True."
else:
    availability_note = "Could not compute -- gsc_data_available not available when this cell ran."

# 3. Position data completeness: zero means no-data, not rank 0.
position_note = ""
if 'df' in locals() and not df.empty and 'gsc_avg_position' in df.columns:
    zero_frac = (df['gsc_avg_position'] == 0).mean()
    position_note = f"Observed: {zero_frac:.1%} of rows have gsc_avg_position == 0 (no-data, excluded from any position feature)."
else:
    position_note = "Could not compute -- gsc_avg_position not available when this cell ran."

data_limitations = {
    'uneven_client_history_depth': {
        'exists': True,
        'description': ("Per-client history depth is not uniform within this slice, so clients with "
                         "thinner history contribute fewer, noisier feature rows than well-covered clients. "
                         f"{history_depth_note}")
    },
    'gsc_availability_gaps': {
        'exists': True,
        'description': ("Not every row has usable GSC data; features built from gsc_clicks/gsc_impressions "
                         "should be filtered on gsc_data_available == True or they'll silently include rows "
                         f"with no real search signal. {availability_note}")
    },
    'position_signal_incomplete': {
        'exists': True,
        'description': (f"gsc_avg_position == 0 means 'no data', not rank zero, and averaging it in unfiltered "
                         f"would corrupt any position-based feature. {position_note}")
    },
    'ga4_zero_fill_before_start': {
        'exists': True,
        'description': ("GA4 columns (ga4_pageviews, ga4_sessions, etc.) are zero-filled for a client before "
                         "that client's ga4_data_start, per flyrank-data. isnull() checks won't catch this since "
                         "the values are 0, not NaN -- any engagement feature must also filter on "
                         "ga4_data_available == True, not just check for missing values.")
    },
    'label_not_yet_derived': {
        'exists': True,
        'description': ("is_declining_label does not exist as a raw column in fact_content_daily_performance -- "
                         "it must be derived (e.g. from a click trend comparison across two halves of the month) "
                         "before it can be used. Whatever threshold or window is chosen for that derivation is a "
                         "modeling assumption, not an observed fact, and should be stated explicitly once written.")
    },
    'single_month_slice': {
        'exists': True,
        'description': ("This contract covers only March 2026 (a mid-panel month chosen deliberately, per the "
                         "task's warning that the final month, June 2026, is a sealed outcome window). Patterns "
                         "here are directional for this one month and may not generalize to other months or to "
                         "seasonal effects outside this window.")
    }
}

for limitation, details in data_limitations.items():
    status = 'Present' if details['exists'] else 'Not Present/Applicable'
    print(f"\nLimitation: {limitation.replace('_', ' ').title()} - {status}")
    if details['exists']:
        print(f"Description: {details['description']}")

print("\n--- Each limitation above is tied to something this notebook actually measured, or flagged as an assumption if it isn't measured yet. ---")

--- Data Limitations Documentation ---


/tmp/ipykernel_32408/62026541.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  days_per_client = df.groupby('client_hash_id')['report_date'].nunique()



Limitation: Uneven Client History Depth - Present
Description: Per-client history depth is not uniform within this slice, so clients with thinner history contribute fewer, noisier feature rows than well-covered clients. Observed: within March 2026, per-client day coverage ranges from 9 to 31 days out of the month; 3 of 55 clients have less than half the median coverage.

Limitation: Gsc Availability Gaps - Present
Description: Not every row has usable GSC data; features built from gsc_clicks/gsc_impressions should be filtered on gsc_data_available == True or they'll silently include rows with no real search signal. Observed: 36.7% of rows in this slice have gsc_data_available == True.

Limitation: Position Signal Incomplete - Present
Description: gsc_avg_position == 0 means 'no data', not rank zero, and averaging it in unfiltered would corrupt any position-based feature. Observed: 1.7% of rows have gsc_avg_position == 0 (no-data, excluded from any position feature).

Limitation: Ga4 Z